# Statistical Analysis Notebook

This notebook has been developed to perform the cell segmentation and counting for statistical analysis associated with the study titled:

**"Protocol for intracardiac delivery and live imaging of NK–tumor interactions in an ex ovo chick embryo model."**

It provides a structured and reproducible workflow for data processing and analysis.

# **INTRODUCTION:**

In the protocol for intracardiac delivery and live imaging of NK–tumor interactions in an ex ovo chick embryo model, segmentation is performed to identify two types of cells: Hoechst-labeled NK cells, which appear as blue fluorescent cells, and GFP-positive neuroblastoma cells, which appear as green fluorescent cells. The segmentation approach vary depending on the signal quality and the imaging time point.

A Python-based analysis scripts are designed for the segmentation and quantification of both cell types. The workflow includes classical image processing techniques as well as deep learning-based methods.

The provided scripts are ready to use and can be executed by simply specifying the dataset path, which is clearly indicated within scripts. There are certain parameters may need to be adjusted depending on the dataset; these parameters and their effects are explained above each script for user guidance.

The scripts are designed to be easily run in the Google Colab environment by uploading the dataset, eliminating the need for local environment setup. Below are the steps to follow in order to run the scripts.

The whole processing requires the following four steps.


1.   Uploading your Images (jpeg FORMAT)
2.   Installing required Packages
3. Pre-processing (background isolation) NOTE: YOU CAN SKIP THIS STEP IF YOUR IMAGES ALREADY HAVE BLACK BACKGROUND
4. CELL SEGMENTATION AND COUNTING







# **STEP 01:**

The first step is to upload your dataset to the Google Colab environment. You can do this by using the left panel (file explorer) in Colab. Click on the folder icon, then use the Upload option to select and upload your images.

# **STEP 02:**

Please run the cells below to install the required libraries and import them to further use in the workflow. You can do this by clicking the Run icon, or by selecting the cell and pressing Shift + Enter.



In [ ]:
!pip install cellpose==2.2.2 #also the 2.2.0 version can be used
!pip install tifffile
!pip install scikit-image
!pip install openCV-python

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# **STEP 03:**

Before segmentation, background removal is performed, as fluorescence image backgrounds can significantly interfere with accurate cell detection. The dataset used in this study contains two types of images based on their background color: images with a blue fluorescence background and images with a green fluorescence background.

To account for differences in color distribution and background noise, two separate preprocessing scripts are provided, each tailored to a specific background type. If your image has a blue background interference, use script in **STEP 3A**. If your image has a green background interference, use script **STEP 3B**.

***NOTE: IF THE IMAGES ALREADY HAVE BLACK BACKGROUND YOU CAN SIMPLY SKIP THIS WHOLE STEP 3 AND DIRECTLY APPLY IMAGE SEGMENTATION BY USING THE STEP 4.***



***STEP 3A:***

This script is designed for images where the background contains a blue haze, and the blue cells have relatively weak signals with possible green interference.

Before running the script, you need to specify the input and output paths in the first two lines of code:



1.   In the img_path variable, enter the full path of your input image.
2.   In the output_folder variable, enter the path of the folder where you want the output results to be saved.

The key parameters have been pre-adjusted for this script. However, if the results are not satisfactory for your dataset, you can fine-tune the parameters using the table provided below.

| Parameter | Default | ↑ Increase Effect | ↓ Decrease Effect | Range |
|---|---|---|---|---|
| `BLUR_KERNEL` | 31 *(odd only)* | Stronger haze removal | Preserves faint signal | 21–51 |
| `BLUE_HUE_MIN` | 75 | Pure blue only | Includes cyan/teal | 65–90 |
| `BLUE_HUE_MAX` | 160 | Includes violet/purple | Pure blue only | 140–165 |
| `BLUE_SAT_MIN` | 45 | Vivid blue only | Catches weak blue ⚠ | 25–65 |
| `BLUE_DOMINANCE` | 25 | Rejects green overlap | Catches faint blue ⚠ | 8–30 |
| `GREEN_DOMINANCE` | 6 | Pure green only | Catches pale green | 4–15 |
| `MIN_BLUE_AREA` | 80 px | Removes noise specks | Keeps small fragments | 40–120 |
| `MIN_FINAL_AREA` | 120 px | Large cells only ⚠ | Keeps small cells + noise | 80–300 |
| `GREEN_SUPPRESS` | 0.40 | More green retained | Purer blue, may darken | 0.25–0.55 |

>
The blue detection is primarily controlled by the hue range (75–160) and the relaxed saturation (≥45) and brightness thresholds (≥80), which help capture weak blue cells.

Additionally, a blue-dominance rule (B > G + 25 and B > R + 25) ensures that even faint blue cells are detected while suppressing non-cell regions.



In [ ]:
# Input and Output Paths
img_path = r"ENTER YOUR INPUT IMAGE PATH HERE" # just copy paste your image path between inverted commas
output_folder = r"ENTER THE PATH OF PATH OF FOLDER WHWERE OUTPUT IMAGE IS SAVED" #just insert your output folder path between inverted commas

os.makedirs(output_folder, exist_ok=True)
bgr = cv2.imread(img_path)
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

# Step 1: Background suppression
blur = cv2.GaussianBlur(rgb, (31, 31), 0)
subtracted = cv2.subtract(rgb, blur)

# Step 2: Enhance contrast
lab = cv2.cvtColor(subtracted, cv2.COLOR_RGB2LAB)
l, a, b = cv2.split(lab)
l = cv2.equalizeHist(l)
lab_eq = cv2.merge([l, a, b])
enhanced = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2RGB)

# Step 3: Convert to HSV
hsv = cv2.cvtColor(enhanced, cv2.COLOR_RGB2HSV)
H, S, V = cv2.split(hsv)
R, G, B = cv2.split(enhanced)

# Step 4: RELAXED BLUE MASK
mask_blue = (
    ((H >= 75) & (H <= 160) & (S >= 45) & (V >= 80)) |      # relaxed saturation and brightness
    ((B > G + 25) & (B > R + 25))                           # relaxed blue dominance
)

mask_green = (
    ((H >= 35) & (H <= 95) & (S >= 20) & (V >= 40)) |
    ((G > R + 6) & (G > B + 6))
)

# Step 5: Combine masks
mask = (mask_blue | mask_green).astype(np.uint8) * 255

# First-pass blue area removal
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_blue.astype(np.uint8))

min_area = 80
clean_blue = np.zeros_like(mask_blue, dtype=bool)

for i in range(1, num_labels):
    if stats[i, cv2.CC_STAT_AREA] >= min_area:
        clean_blue[labels == i] = True

mask_blue = clean_blue
mask = ((mask_blue | mask_green).astype(np.uint8)) * 255

# Morphological cleaning
kernel = np.ones((3, 3), np.uint8)
mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel)
mask_clean = cv2.dilate(mask_clean, kernel, iterations=1)

# Second-pass area filtering
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_clean)

final_mask = np.zeros_like(mask_clean)
min_final_area = 120

for i in range(1, num_labels):
    if stats[i, cv2.CC_STAT_AREA] >= min_final_area:
        final_mask[labels == i] = 255

mask_clean = final_mask

# Step 6: Apply mask
result = cv2.bitwise_and(rgb, rgb, mask=mask_clean)
black_background = np.zeros_like(rgb)
final = np.where(result > 0, result, black_background)

# Remove green tint from blue cells
mask_blue = mask_blue.astype(bool)
final_corrected = final.copy()
R_f, G_f, B_f = cv2.split(final_corrected)

only_blue = mask_blue & (~mask_green)
G_f[only_blue] = (G_f[only_blue] * 0.40).astype(np.uint8)
R_f[only_blue] = (R_f[only_blue] * 0.85).astype(np.uint8)

final_corrected = cv2.merge([R_f, G_f, B_f])

# Step 7: Display
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(rgb)
plt.title("Original Image")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(final_corrected)
plt.title("Output image")
plt.axis('off')
plt.show()

# Step 8: Save
filename = os.path.basename(img_path)
output_path = os.path.join(output_folder, filename)
cv2.imwrite(output_path, cv2.cvtColor(final_corrected, cv2.COLOR_RGB2BGR))

print("Output saved:", output_path)

***STEP 3B:***

This script is optimized for images where the background contains green fluorescence artifacts and faint green signals that must be preserved. Before running the script, you need to specify the input and output paths in the first two lines of code:



1.   In the img_path variable, enter the full path of your input image.
2.   In the output_folder variable, enter the path of the folder where you want the output results to be saved.

The key parameters have been pre-adjusted for this script. However, if the results are not satisfactory for your dataset, you can fine-tune the parameters using the table provided below.

| Parameter | Default | ↑ Increase Effect | ↓ Decrease Effect | Range |
|---|---|---|---|---|
| `BLUR_KERNEL` | 41 *(odd only)* | Stronger background removal | Preserves faint signal | 21–51 |
| `GREEN_HUE_MIN` | 25 | Excludes yellow-green | Includes yellow/orange | 15–45 |
| `GREEN_HUE_MAX` | 105 | Includes cyan-green | Pure green only | 90–120 |
| `GREEN_SAT_MIN` | 5 | Vivid green only | Catches very faint green ⚠ | 5–40 |
| `GREEN_BRIGHT_MIN` | 15 | Rejects dark regions | Catches dim cells ⚠ | 10–50 |
| `GREEN_DOMINANCE` | 5 | Pure green only | Catches pale/weak green ⚠ | 3–20 |
| `GREEN_INTENSITY` | 14 | Rejects very dark pixels | Catches extremely dim cells | 10–40 |
| `MIN_CELL_SIZE` | 3 px | Removes noise specks | Keeps tiny fragments ⚠ | 3–100 |

The green-background preprocessing script uses flexible thresholds to preserve even faint green fluorescence. The primary HSV mask uses a wide hue range (25–105) with relaxed saturation (>5) and value (>15) limits, allowing detection of low-intensity green cells. The secondary green-intensity mask relies on green dominance (G > R + 5 and G > B + 5) to keep genuine cells while suppressing background noise.

The script also uses CLAHE enhancement (clipLimit = 3.0) to amplify visibility of dim and small green cells.

In [ ]:
# Input and Output Paths
img_path = r"ENTER YOUR INPUT IMAGE PATH"
output_folder = r"ENTER THE OUTPUT FOLDER PATH WHERE THE OUTPUT IMAGE TO BE SAVED"

os.makedirs(output_folder, exist_ok=True)
bgr = cv2.imread(img_path)
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

# Step 1: Background suppression
blur = cv2.GaussianBlur(rgb, (41, 41), 0)
subtracted = cv2.subtract(rgb, blur)

# Step 2: CLAHE
lab = cv2.cvtColor(subtracted, cv2.COLOR_RGB2LAB)
l, a, b = cv2.split(lab)
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
l_clahe = clahe.apply(l)
lab_clahe = cv2.merge([l_clahe, a, b])
enhanced = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)

# Step 3: Convert to HSV
hsv = cv2.cvtColor(enhanced, cv2.COLOR_RGB2HSV)
H, S, V = cv2.split(hsv)

# Step 4: RELAXED GREEN MASK (cells reappear)
mask_hsv = (
    (H > 25) & (H < 105) &
    (S > 5) &        # saturation
    (V > 15)          # brightness
).astype(np.uint8) * 255

# Step 5: Green channel detector
R, G, B = cv2.split(enhanced)

mask_green = (
    (G > R + 5) &
    (G > B + 5) &
    (G > 14)
).astype(np.uint8) * 255

# KEY FIX
mask_combined = cv2.bitwise_or(mask_hsv, mask_green)

# Step 6: Light cleaning
kernel = np.ones((3, 3), np.uint8)
mask_clean = cv2.morphologyEx(mask_combined, cv2.MORPH_OPEN, kernel)

# Step 7: Connected components
num_labels, labels = cv2.connectedComponents(mask_clean)

min_size = 3   # cell size
final_mask = np.zeros_like(mask_clean)

for label in range(1, num_labels):
    component_size = np.sum(labels == label)
    if component_size >= min_size:
        final_mask[labels == label] = 255

# Step 8: Apply mask
result = cv2.bitwise_and(rgb, rgb, mask=final_mask)
black_background = np.zeros_like(rgb)
final = np.where(result > 0, result, black_background)

# Step 9: Show results
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(rgb)
plt.title("Original Image")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(final)
plt.title("Green Cells Detected")
plt.axis('off')
plt.show()

# Step 10: Save
filename = os.path.basename(img_path)
output_path = os.path.join(output_folder, filename)
cv2.imwrite(output_path, cv2.cvtColor(final, cv2.COLOR_RGB2BGR))

print("Output saved:", output_path)

# SEGMENTATION:
For the segmentation following two methods are used.


1.   Otsu Thresholding Method
2.   Cellpose

# STEP 04:

Once the background has been removed from the images, you can proceed with segmentation and proximity analysis using the Step 04 scripts. You may try both methods and choose the one that provides better results for your data. It is recommended to first try the Otsu-based method, as it does not require GPU support and is faster to execute, whereas the Cellpose-based method requires GPU access and may take longer to run.


***STEP 4A:***

To run this script, go to the 0. USER CONFIG section and provide:

the path of the input image with the background already removed, and
 the output directory where the result images will be saved.

The output includes:

the total count of blue cells and green cells, and the proximity analysis based on the selected radius, showing how many blue cells are present around each green cell.

The results are also generated in table format for easier interpretation and handling. In addition, you need to set the radius value according to your requirement in Section 5: px Radius Analysis of the code.

#### Adjustable Parameters in Otsu Thresholding script

| Category | Parameter | Current Value | Description | Effect if Adjusted |
|----------|-----------|--------------|-------------|--------------------|
| Thresholding | `Otsu Threshold` | Automatic | Separates cells from background | Changing thresholding strategy can alter detected cell regions |
| Watershed Seed Detection | `min_distance` | `4` | Minimum distance between detected cell centers | Larger value reduces over-segmentation of nearby cells |
| Noise Removal | `len(xs) < 5` | `5` pixels | Minimum pixel count required for a detected cell region | Increasing removes very small detected objects |
| Green Cell Filter | `area < 20`, `w < 6`, `h < 6` | `20, 6, 6` | Removes small green cell fragments | Increasing thresholds removes small green blobs |
| Blue Cell Filter | `area < 40`, `w < 20`, `h < 20` | `40, 20, 20` | Removes small blue fragments | Larger values keep only larger blue cells |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from scipy import ndimage as ndi
import os
import pandas as pd

# ---------------------------
# 0. USER CONFIG
# ---------------------------
IMAGE_PATH = "ENTER YOUR INPUT IMAGE PATH (BACKGROUND REMOVED)"
OUTPUT_DIR = "ENTER YOUR OUTPUT FOLDER PATH"
os.makedirs(OUTPUT_DIR, exist_ok=True)

base_name = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
OUTPUT_IMAGE = os.path.join(OUTPUT_DIR, f"{base_name}_output.jpeg")
OUTPUT_TABLE = os.path.join(OUTPUT_DIR, f"{base_name}_output_table.jpeg")

# ---------------------------
# 1. LOAD IMAGE
# ---------------------------
img = cv2.imread(IMAGE_PATH)

# Check if image was loaded successfully
if img is None:
    raise FileNotFoundError(f"Error: Image not found at {IMAGE_PATH}. Please check the path and file name.")

img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
R, G, B = cv2.split(img)

# ---------------------------
# 2. PRE-PROCESSING (SAME AS BEFORE)
# ---------------------------
gray_input = np.maximum(G, B)
gray_blur = cv2.GaussianBlur(gray_input, (5, 5), 0)
gray_norm = cv2.normalize(gray_blur, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# green enhancement (you might reuse later; not strictly needed for segmentation)
G_boost = cv2.normalize(G.astype(float) * 3.0, None, 0, 255, cv2.NORM_MINMAX)
green_diff = cv2.normalize((G_boost - B).clip(0), None, 0, 255, cv2.NORM_MINMAX)
green_diff_clahe = green_diff

# ---------------------------
# 3. SIMPLE SEGMENTATION (OTSU + WATERSHED)  **INSTEAD OF CELLPOSE**
# ---------------------------

# Global Otsu threshold works very well for black background + bright cells
_, binary = cv2.threshold(
    gray_norm, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

# Light cleaning
mask_clean = cv2.morphologyEx(
    binary,
    cv2.MORPH_OPEN,
    np.ones((3, 3), np.uint8)
)

# Distance transform
dist = ndi.distance_transform_edt(mask_clean)

# Local maxima as seeds
coords = peak_local_max(
    dist,
    min_distance=3,       # small distance since cells are small and sparse
    labels=mask_clean,
    exclude_border=False
)

local_max = np.zeros_like(dist, dtype=bool)
if len(coords) > 0:
    local_max[tuple(coords.T)] = True

markers = ndi.label(local_max)[0]

# Watershed → final labels_ws (1 label per cell or cluster)
labels_ws = watershed(-dist, markers, mask=mask_clean)

# ---------------------------
# 4. SUB-CELL COLOR SPLITTING
# ---------------------------
contour_img = img.copy()

green_cells = []
blue_cells = []
green_count = 0
blue_count = 0

kernel = np.ones((2, 2), np.uint8)  # small kernel, keeps tiny blobs

# -----------------------------------------
# CONTOUR DRAWING FUNCTION
# -----------------------------------------
def draw_perfect_contour(mask_binary, color):
    m = (mask_binary.astype(np.uint8) * 255)

    m = cv2.bilateralFilter(m, d=5, sigmaColor=30, sigmaSpace=30)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    edges = cv2.Canny(m, 30, 100)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        if len(cnt) < 5:
            continue
        epsilon = 0.01 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)
        cv2.polylines(contour_img, [approx], True, color, 1, cv2.LINE_AA)


# ---------------------------
# GREEN + BLUE LOOP
# ---------------------------
for cell_id in range(1, labels_ws.max() + 1):

    mask = (labels_ws == cell_id)
    ys, xs = np.where(mask)

    # extremely tiny regions → skip
    if len(xs) < 5:
        continue

    G_cell = G[mask]
    B_cell = B[mask]

    # classification: softer but still meaningful
    green_full = (G_cell > B_cell * 1.10).astype(np.uint8)
    blue_full  = (B_cell > G_cell * 1.10).astype(np.uint8)

    gf = np.zeros_like(mask, dtype=np.uint8)
    bf = np.zeros_like(mask, dtype=np.uint8)
    gf[mask] = green_full
    bf[mask] = blue_full

    gf = cv2.morphologyEx(gf, cv2.MORPH_OPEN, kernel)
    bf = cv2.morphologyEx(bf, cv2.MORPH_OPEN, kernel)

    n_g, lab_g = cv2.connectedComponents(gf.astype(np.uint8))
    n_b, lab_b = cv2.connectedComponents(bf.astype(np.uint8))

    # --- GREEN CELLS ---
    for gid in range(1, n_g):
        ys2, xs2 = np.where(lab_g == gid)
        area = len(xs2)
        x1, x2 = xs2.min(), xs2.max()
        y1, y2 = ys2.min(), ys2.max()
        w = x2 - x1 + 1
        h = y2 - y1 + 1

        if area < 20 and w < 6 and h < 6:
            continue

        green_count += 1
        cx, cy = int(xs2.mean()), int(ys2.mean())
        green_cells.append((f"G{green_count}", cx, cy))

        cv2.putText(contour_img, f"G{green_count}", (cx+3, cy-3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 0), 1)

        mask_bin = (lab_g == gid).astype(np.uint8)
        draw_perfect_contour(mask_bin, (255, 255, 0))  # yellow

    # --- BLUE CELLS ---
    for bid in range(1, n_b):
        ys2, xs2 = np.where(lab_b == bid)
        area = len(xs2)
        x1, x2 = xs2.min(), xs2.max()
        y1, y2 = ys2.min(), ys2.max()
        w = x2 - x1 + 1
        h = y2 - y1 + 1

        if area < 22 and w < 7 and h < 7:
            continue

        blue_count += 1
        cx, cy = int(xs2.mean()), int(ys2.mean())
        blue_cells.append((f"B{blue_count}", cx, cy))

        mask_bin = (lab_b == bid).astype(np.uint8)
        draw_perfect_contour(mask_bin, (255, 0, 0))  # blue

# ---------------------------
# 5. PX RADIUS ANALYSIS
# ---------------------------
RADIUS = 139 # THIS VALUE CAN BE SET ACCORDING TO THE DESIRED PROXIMITY ANALYSIS
table_data = []

for label_g, gx, gy in green_cells:
    count_blue = 0
    cv2.circle(contour_img, (gx, gy), RADIUS, (0, 255, 255), 1)
    for label_b, bx, by in blue_cells:
        if np.sqrt((gx - bx) ** 2 + (gy - by) ** 2) <= RADIUS:
            count_blue += 1
    table_data.append([label_g, count_blue])

df = pd.DataFrame(table_data, columns=["Green Cell", "Blue Cells in 139px"])
print(df)

# ---------------------------
# 6. SUMMARY TEXT
# ---------------------------
summary = f"Green: {green_count} | Blue: {blue_count} | Total: {green_count + blue_count}"
cv2.putText(contour_img, summary, (10, 25),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

# ---------------------------
# 7. SAVE OUTPUT
# ---------------------------
cv2.imwrite(OUTPUT_IMAGE, cv2.cvtColor(contour_img, cv2.COLOR_RGB2BGR))
print("Saved IMAGE:", OUTPUT_IMAGE)

plt.figure(figsize=(10, 10))
plt.imshow(contour_img)
plt.axis("off")
plt.title("Final Output")
plt.show()

# ---------------------------
# SAVE TABLE IMAGE
# ---------------------------
if not df.empty:
    fig, ax = plt.subplots(figsize=(4, len(df) * 0.6))
    ax.axis('off')
    tbl = ax.table(cellText=df.values, colLabels=df.columns,
                   cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    tbl.scale(1, 1.4)

    plt.title(f"Blue Cells within {RADIUS} px of Green Cells")
    fig.savefig(OUTPUT_TABLE, dpi=300, bbox_inches='tight')
    print("Saved TABLE:", OUTPUT_TABLE)
else:
    print("No green cells found, skipping table generation.")


**The script below is designed for images containing only green cells (CONTROL GROUP) and is used for their segmentation and counting. This script does not include proximity analysis.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
from skimage.filters import threshold_otsu
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy import ndimage as ndi
import os

IMAGE_PATH = "ENTER YOUR INPUT IMAGE PATH (BACKGROUND REMOVED)" #input image path here
OUTPUT_DIR = "ENTER YOUR OUTPUT FOLDER PATH" #output folder path
os.makedirs(OUTPUT_DIR, exist_ok=True)

base_name = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
OUTPUT_IMAGE = os.path.join(OUTPUT_DIR, f"{base_name}_GREEN_output.png")

# 1. LOAD IMAGE
img = cv2.imread(IMAGE_PATH)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
R, G, B = cv2.split(img)

# 2. PRE-PROCESSING FOR GREEN
gray_input = G
gray_blur = cv2.GaussianBlur(gray_input, (5, 5), 0)
gray_norm = cv2.normalize(gray_blur, None, 0, 255, cv2.NORM_MINMAX)

G_boost = cv2.normalize(G.astype(float) * 3.0, None, 0, 255, cv2.NORM_MINMAX)
green_diff = cv2.normalize((G_boost - B).clip(0), None, 0, 255, cv2.NORM_MINMAX)

clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
green_diff_clahe = clahe.apply(green_diff.astype(np.uint8))

# 3. SIMPLE SEGMENTATION (NO CELLPOSE)
th = threshold_otsu(green_diff_clahe)
bw = (green_diff_clahe > th).astype(np.uint8)

kernel = np.ones((3, 3), np.uint8)
bw_clean = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel)

num_labels, labels_ws = cv2.connectedComponents(bw_clean)


# CONTOURING
contour_img = img.copy()

def draw_perfect_contour(mask_binary, color):
    m = (mask_binary.astype(np.uint8) * 255)
    m = cv2.bilateralFilter(m, d=5, sigmaColor=30, sigmaSpace=30)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8))
    edges = cv2.Canny(m, 30, 100)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        if len(cnt) < 5:
            continue
        epsilon = 0.01 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)
        cv2.polylines(contour_img, [approx], True, color, 1, cv2.LINE_AA)


# 4. GREEN CELL DETECTION, BIG-CELL SPLITTING AND TINY NOISE REMOVAL
green_cells = []
green_count = 0

for cell_id in range(1, num_labels):

    mask = (labels_ws == cell_id)
    if mask.sum() < 50:
        continue

    G_cell = G[mask].astype(float)
    B_cell = B[mask].astype(float)
    if np.mean(G_cell) < np.mean(B_cell) * 1.1:
        continue

    # BIG-CELL SPLITTING
    dist = ndi.distance_transform_edt(mask)

    coords = peak_local_max(
        dist,
        min_distance=1,
        labels=mask,
        footprint=np.ones((25, 25)),
    )

    local_max_mask = np.zeros_like(dist, dtype=bool)
    if len(coords) > 0:
        local_max_mask[tuple(coords.T)] = True

    markers = ndi.label(local_max_mask)[0]
    splitted = watershed(-dist, markers, mask=mask)

    # Loop over sub-cells
    for sub_id in range(1, splitted.max() + 1):

        submask = (splitted == sub_id)
        ys, xs = np.where(submask)


        # NOISE REMOVAL FILTERS
        # 1. Minimum area
        if submask.sum() < 25:
            continue

        # 2. Minimum bounding box size
        h = ys.max() - ys.min()
        w = xs.max() - xs.min()
        if h < 10 and w < 10:
            continue

        # 3. Intensity must be bright enough
        G_sub = G[submask].astype(float)
        if np.mean(G_sub) < 50:
            continue

        # ACCEPTED CELL

        green_count += 1
        cx, cy = int(xs.mean()), int(ys.mean())
        green_cells.append((f"G{green_count}", cx, cy))

        cv2.putText(contour_img, f"G{green_count}", (cx+3, cy-3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,0), 1)

        draw_perfect_contour(submask.astype(np.uint8), (255,255,0))

# 5. SUMMARY
summary = f"Green Cells: {green_count}"
cv2.putText(contour_img, summary, (10,25),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)

print(summary)


# 6. SAVE OUTPUT
cv2.imwrite(OUTPUT_IMAGE, cv2.cvtColor(contour_img, cv2.COLOR_RGB2BGR))
print("Saved:", OUTPUT_IMAGE)

plt.figure(figsize=(10,10))
plt.imshow(contour_img)
plt.axis("off")
plt.title("FINAL OUTPUT IMAGE")
plt.show()

***STEP 4B:***

To run this script, go to the 0. USER CONFIG section and provide:

the path of the input image with the background already removed, and the output directory where the result images will be saved.

The output includes:

the total count of blue cells and green cells, and the proximity analysis based on the selected radius, showing how many blue cells are present around each green cell.

The results are also generated in table format for easier interpretation and handling. In addition, you need to set the radius value according to your requirement in Section 6: px Radius Analysis of the code.

#### Adjustable Parameters in Cellpose script

| Category | Parameter | Current Value | Description | Effect if Adjusted |
|----------|-----------|--------------|-------------|--------------------|
| Cellpose | `model_type` | `cyto2` | Pretrained Cellpose model used for segmentation | Changing to `cyto` or `nuclei` alters the type of cellular structures detected |
| Cellpose | `diameter` | `None` | Expected cell diameter in pixels | Setting a value (e.g., `30–50`) helps Cellpose detect correct cell size |
| Cellpose | `min_size` | `5` | Minimum size of detected objects kept as cells | Increasing removes small noise but may miss very small cells |
| Cellpose | `rescale` | `0.75` | Image scaling factor before segmentation | Lower values speed up processing but may reduce segmentation accuracy |
| Color Classification (Green) | `G_cell > B_cell * 1.15` | `1.15` | Threshold for classifying pixels as green-dominant cells | Higher value makes green detection stricter |
| Color Classification (Blue) | `B_cell > G_cell * 1.05` | `1.05` | Threshold for classifying pixels as blue-dominant cells | Higher value makes blue detection stricter |

In [ ]:
from cellpose import models
import matplotlib.pyplot as plt
import numpy as np
import cv2
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from scipy import ndimage as ndi
import os
import pandas as pd

# ---------------------------
# 0. USER CONFIG
# ---------------------------
IMAGE_PATH = "ENTER YOUR INPUT IMAGE PATH (BACKGROUND REMOVED)"
OUTPUT_DIR = "ENTER YOUR OUTPUT FOLDER PATH"
os.makedirs(OUTPUT_DIR, exist_ok=True)

base_name = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
OUTPUT_IMAGE = os.path.join(OUTPUT_DIR, f"{base_name}_output.jpeg")
OUTPUT_TABLE = os.path.join(OUTPUT_DIR, f"{base_name}_output_table.jpeg")

# ---------------------------
# 1. LOAD IMAGE
# ---------------------------
img = cv2.imread(IMAGE_PATH)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
R, G, B = cv2.split(img)

# ---------------------------
# 2. PRE-PROCESSING
# ---------------------------
gray_input = np.maximum(G, B)
gray_blur = cv2.GaussianBlur(gray_input, (5, 5), 0)
gray_norm = cv2.normalize(gray_blur, None, 0, 255, cv2.NORM_MINMAX)

G_boost = cv2.normalize(G.astype(float) * 3.0, None, 0, 255, cv2.NORM_MINMAX)
green_diff = cv2.normalize((G_boost - B).clip(0), None, 0, 255, cv2.NORM_MINMAX)

clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
green_diff_clahe = clahe.apply(green_diff.astype(np.uint8))

# ---------------------------
# 3. CELLPOSE
# ---------------------------
try:
    model = models.Cellpose(gpu=True, model_type='cyto2')
except:
    model = models.Cellpose(gpu=False, model_type='cyto2')

masks, flows, styles, diams = model.eval(
    gray_norm, channels=[0, 0], diameter=None, min_size=5, rescale=0.75
)

# ---------------------------
# 4. WATERSHED
# ---------------------------
mask_clean = cv2.morphologyEx((masks > 0).astype(np.uint8),
                              cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
dist = ndi.distance_transform_edt(mask_clean)

coords = peak_local_max(dist, min_distance=3, labels=mask_clean, exclude_border=False)
local_max = np.zeros_like(dist, dtype=bool)
if len(coords) > 0:
    local_max[tuple(coords.T)] = True

markers = ndi.label(local_max)[0]
labels_ws = watershed(-dist, markers, mask=mask_clean)

# ---------------------------
# 5. SUB-CELL COLOR SPLITTING
# ---------------------------
contour_img = img.copy()

green_cells = []
blue_cells = []
green_count = 0
blue_count = 0

kernel = np.ones((3,3), np.uint8)

# -----------------------------------------
# NEW TIGHT + SMOOTH + ENHANCED CONTOURING
# -----------------------------------------
def draw_perfect_contour(mask_binary, color):

    # ensure uint8 mask
    m = (mask_binary.astype(np.uint8) * 255)

    # 1. Bilateral filter: smooth edges but keep tight shape
    m = cv2.bilateralFilter(m, d=5, sigmaColor=30, sigmaSpace=30)

    # 2. Light closing with small kernel (DOES NOT expand boundary)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8))

    # 3. Canny edges for crisp boundary
    edges = cv2.Canny(m, 30, 100)

    # 4. Find contours from edges
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 5. Simplify contour + anti-aliased drawing
    for cnt in contours:
        if len(cnt) < 5:
            continue

        epsilon = 0.01 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)

        # Anti-aliased enhanced contour
        cv2.polylines(contour_img, [approx], True, color, 1, cv2.LINE_AA)


# ---------------------------
# GREEN + BLUE LOOP
# ---------------------------
for cell_id in range(1, labels_ws.max() + 1):

    mask = (labels_ws == cell_id)
    ys, xs = np.where(mask)

    if len(xs) < 10:
        continue

    G_cell = G[mask]
    B_cell = B[mask]

    green_full = (G_cell > B_cell * 1.05).astype(np.uint8)
    blue_full  = (B_cell > G_cell * 1.00).astype(np.uint8)

    gf = np.zeros_like(mask, dtype=np.uint8)
    bf = np.zeros_like(mask, dtype=np.uint8)

    gf[mask] = green_full
    bf[mask] = blue_full

    gf = cv2.morphologyEx(gf, cv2.MORPH_OPEN, kernel)
    bf = cv2.morphologyEx(bf, cv2.MORPH_OPEN, kernel)

    n_g, lab_g = cv2.connectedComponents(gf.astype(np.uint8))
    n_b, lab_b = cv2.connectedComponents(bf.astype(np.uint8))

    # --- GREEN CELLS ---
    for gid in range(1, n_g):
        ys2, xs2 = np.where(lab_g == gid)
        area = len(xs2)

        x1, x2 = xs2.min(), xs2.max()
        y1, y2 = ys2.min(), ys2.max()
        w = x2 - x1 + 1
        h = y2 - y1 + 1

        if area < 22 and w < 6 and h < 6:
            continue

        green_count += 1
        cx, cy = int(xs2.mean()), int(ys2.mean())
        green_cells.append((f"G{green_count}", cx, cy))

        cv2.putText(contour_img, f"G{green_count}", (cx+3, cy-3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255,255,0), 1)

        mask_bin = (lab_g == gid).astype(np.uint8)
        draw_perfect_contour(mask_bin, (255,255,0))  # yellow

    # --- BLUE CELLS ---
    for bid in range(1, n_b):
        ys2, xs2 = np.where(lab_b == bid)
        area = len(xs2)

        x1, x2 = xs2.min(), xs2.max()
        y1, y2 = ys2.min(), ys2.max()
        w = x2 - x1 + 1
        h = y2 - y1 + 1

        if area < 4 and w < 2 and h < 2:
            continue

        blue_count += 1
        cx, cy = int(xs2.mean()), int(ys2.mean())
        blue_cells.append((f"B{blue_count}", cx, cy))

        mask_bin = (lab_b == bid).astype(np.uint8)
        draw_perfect_contour(mask_bin, (255,0,0))  # blue

# ---------------------------
# 6. PX RADIUS ANALYSIS
# ---------------------------
RADIUS = 145  #JUST ENTER YOUR DESIRED RADIUS VALUE HERE
table_data = []

for label_g, gx, gy in green_cells:

    count_blue = 0
    cv2.circle(contour_img, (gx, gy), RADIUS, (0, 255, 255), 1)

    for label_b, bx, by in blue_cells:
        if np.sqrt((gx - bx)**2 + (gy - by)**2) <= RADIUS:
            count_blue += 1

    table_data.append([label_g, count_blue])

df = pd.DataFrame(table_data, columns=["Green Cell", "Blue Cells in 145 px"])
print(df)

# ---------------------------
# 7. SUMMARY TEXT
# ---------------------------
summary = f"Green: {green_count} | Blue: {blue_count} | Total: {green_count + blue_count}"
cv2.putText(contour_img, summary, (10,25),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)

# ---------------------------
# 8. SAVE OUTPUT
# ---------------------------
cv2.imwrite(OUTPUT_IMAGE, cv2.cvtColor(contour_img, cv2.COLOR_RGB2BGR))
print("Saved IMAGE:", OUTPUT_IMAGE)

plt.figure(figsize=(10,10))
plt.imshow(contour_img)
plt.axis("off")
plt.title("Final Output")
plt.show()

# ---------------------------
# SAVE TABLE IMAGE
# ---------------------------
fig, ax = plt.subplots(figsize=(4, len(df)*0.6))
ax.axis('off')
tbl = ax.table(cellText=df.values, colLabels=df.columns,
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1,1.4)

plt.title(f"Blue Cells within {RADIUS} px of Green Cells")
fig.savefig(OUTPUT_TABLE, dpi=300, bbox_inches='tight')
print("Saved TABLE:", OUTPUT_TABLE)
